### Imports

In [ ]:
%load_ext autoreload
%autoreload 2

# Internal import
import deep_learning_land_use_classification.config as config
from deep_learning_land_use_classification.dataset import get_single_label_data
import deep_learning_land_use_classification.evaluation as evaluation
import deep_learning_land_use_classification.wanddb_helpers as wh

# External imports
import torch
import torch.nn as nn
from torchvision import transforms
from torchvision.models import resnet50, ResNet50_Weights
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import wandb

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
# Get data
train_df, test_df, class_names, num_classes = get_single_label_data()

In [ ]:
# Start a new wandb run to track this script.
run = wh.init_run(
    task="single",
    architecture="resnet50",
    num_classes=num_classes,
    loss="CrossEntropyLoss",
    epochs=5,
    batch_size=32,
    learning_rate=1e-4,
    optimizer="Adam",
    pretrained=True,
    pretraining_dataset="ImageNetV2",
    pretraining_source="torchvision",
    weights="IMAGENET1K_V2",
    model_name=None,
    augmentation=False,
    early_stopping=True,
    patience=2,
    min_delta=0.001
)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\tomer\_netrc.
wandb: Currently logged in as: tomer-peled (sstaheli52-wageningen-university-and-research) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


### Resize, transform and normalize dataset

In [6]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # handles inconsistent sizes
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406], # standard ImageNet mean
        std =[0.229, 0.224, 0.225] # standard ImageNet std
    )
])

### Get training and test dataset, as well as dataset loaders

In [26]:
class SingleLabelDataset(Dataset):
    def __init__(self, df, class_names, transform=None):
        self.df = df.reset_index(drop=True)
        self.class_names = class_names
        self.transform = transform
        self.label_to_idx = {label: i for i, label in enumerate(class_names)}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["path"]).convert("RGB")
        
        if self.transform:
            image = self.transform(image)
        
        label = self.label_to_idx[row["label"]]
        return image, torch.tensor(label, dtype=torch.long)

train_dataset = SingleLabelDataset(train_df, class_names, transform)
test_dataset  = SingleLabelDataset(test_df, class_names, transform)

train_loader = DataLoader(train_dataset, batch_size=run.config.batch_size, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=run.config.batch_size, shuffle=False)

### Initiate model

In [27]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print ("Using device:", device)

model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)

# Replace final layer
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

Using device: cuda


In [28]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

In [ ]:
wh.log_model_summary(run, model)

### Train model

In [30]:
def train(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [31]:
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item()

    return total_loss / len(loader)

In [ ]:
# Early stopping class
class EarlyStopper:
    def __init__(self, patience=2, min_delta=0.001):
        self.patience = patience      # epochs to wait after last improvement
        self.min_delta = min_delta    # minimum change to count as improvement
        self.counter = 0
        self.best_loss = float('inf')
        self.best_weights = None

    def step(self, val_loss, model):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
            # Save a copy of the best weights
            self.best_weights = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            self.counter += 1

        return self.counter >= self.patience  # True = stop training

    def restore_best_weights(self, model):
        if self.best_weights:
            model.load_state_dict(self.best_weights)

In [ ]:
epochs = run.config.epochs
early_stopper = EarlyStopper(patience=run.config.patience, min_delta=run.config.min_delta)

for epoch in range(epochs):
    train_loss = train(model, train_loader, optimizer, criterion)
    test_loss  = evaluate(model, test_loader, criterion)

    print(f"Epoch {epoch+1}/{epochs}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Test Loss:  {test_loss:.4f}")
    
    precision, recall, f1, p_macro, r_macro, f1_macro = evaluation.compute_accuracy_metrics_singlelabel(
        model, test_loader, device
    )
    class_metrics = wandb.Table(columns=["class", "precision", "recall", "f1"])
    for i, class_name in enumerate(class_names):
        class_metrics.add_data(class_name, float(precision[i]), float(recall[i]), float(f1[i]))
        print(f"{class_name:15s} | Precision: {precision[i]:.3f} | Recall: {recall[i]:.3f} | F1: {f1[i]:.3f}")

    wh.log_epoch(run, epoch, train_loss, test_loss,
             precision, recall, f1, p_macro, r_macro, f1_macro, class_names)

    # --- Early stopping check ---
    if early_stopper.step(test_loss, model):
        print(f"Early stopping triggered at epoch {epoch+1}. Best test loss: {early_stopper.best_loss:.4f}")
        break

# Restore the weights from the best epoch
early_stopper.restore_best_weights(model)
print("Restored best model weights.")

run.finish()

Epoch 1/5
Train Loss: 0.3616
Test Loss:  0.1459
(420,) (420,)
[10 11  0 13  6] [10 11  0 13  6]
agricultural    | Precision: 1.000 | Recall: 1.000 | F1: 1.000
airplane        | Precision: 1.000 | Recall: 1.000 | F1: 1.000
baseballdiamond | Precision: 1.000 | Recall: 1.000 | F1: 1.000
beach           | Precision: 1.000 | Recall: 1.000 | F1: 1.000
buildings       | Precision: 1.000 | Recall: 0.958 | F1: 0.979
chaparral       | Precision: 1.000 | Recall: 1.000 | F1: 1.000
denseresidential | Precision: 1.000 | Recall: 0.611 | F1: 0.759
forest          | Precision: 1.000 | Recall: 1.000 | F1: 1.000
freeway         | Precision: 1.000 | Recall: 0.944 | F1: 0.971
golfcourse      | Precision: 1.000 | Recall: 1.000 | F1: 1.000
harbor          | Precision: 1.000 | Recall: 1.000 | F1: 1.000
intersection    | Precision: 1.000 | Recall: 0.944 | F1: 0.971
mediumresidential | Precision: 0.609 | Recall: 0.933 | F1: 0.737
mobilehomepark  | Precision: 0.963 | Recall: 0.963 | F1: 0.963
overpass        | P

macro/f1,▁▃█████
macro/precision,▁▂█████
macro/recall,██▂▂▄▂▁
test_loss,██▁▁▁▁▇▂▂▁▁▁▁▁▁▂▂
train_loss,██▂▂▁▁█▂▂▁▁▁▁▁▁▁▁
macro/f1,0.96119
macro/precision,0.96994
macro/recall,0.96053
test_loss,0.13244
train_loss,0.04292
